# trainer-class-skeleton composite — cx28: Trainer.fit walks a DataLoader: one step per batch, B batches per epoch

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `trainer-class-skeleton`, `dataloader-batching`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "trainer-class-skeleton"
DD_ATOM_IDS = ["trainer-class-skeleton", "dataloader-batching"]
DD_SUBTOPICS = ["Trainer: Trainer class skeleton", "PyTorch: DataLoader batching"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

The Trainer's `fit(n_epochs)` method is where the OO skeleton meets PyTorch's data-loading machinery. Each epoch iterates the `DataLoader`, calling `training_step` on every batch. Two atoms compose:

1. **trainer-class-skeleton** — the `Trainer.fit` method that walks epochs and batches.
2. **dataloader-batching** — `DataLoader(TensorDataset(...), batch_size=B, shuffle=...)` yields batches of size `B`. `len(dataloader) == ceil(N / B)` (or `N // B` if `drop_last=True`).

**Anatomy of `fit`.**
```python
def fit(self, n_epochs):
    for epoch in range(n_epochs):
        for batch in self.train_loader:        # dataloader-batching.
            self.training_step(batch)          # trainer-class-skeleton.
```

**The step count is deterministic.** After `fit(n_epochs)` on a loader with `len(loader) = B_count` batches: `trainer.step == n_epochs * B_count`. That equation is what the test pins down — any off-by-one in the loop (one extra batch, skipping the first, etc.) breaks it.

### Composite Exercise — Trainer.fit walks a DataLoader: one step per batch, B batches per epoch

**Atoms exercised together**: `trainer-class-skeleton`, `dataloader-batching`

Implement `cx28_make_trainer_with_fit()` returning a `LoaderTrainer` class.

Required structure:
- `LoaderTrainer.__init__(self, model, optimizer, loss_fn, train_loader)`:
  - Store the four args. `self.step = 0`. `self.epoch = 0`. `self.history = []`.
- `LoaderTrainer.training_step(self, batch)`:
  - Same as cx27: forward → scalar loss → zero_grad → backward → step → counter++ → history append → return scalar loss.
- `LoaderTrainer.fit(self, n_epochs)`:
  - For each of `n_epochs` epochs:
    - Iterate `for batch in self.train_loader`, call `self.training_step(batch)`.
    - Increment `self.epoch += 1` at the end of the epoch.
  - Return `self.history`.

The test verifies:
- `len(history) == n_epochs * len(train_loader)` (step count equation).
- `trainer.step == n_epochs * len(train_loader)` and `trainer.epoch == n_epochs`.
- Each `batch` yielded by the loader has the correct batch_size (last batch may be smaller if `drop_last=False`).
- Loss decreases across epochs (regression converges).

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx28_make_trainer_with_fit():
    """Return a LoaderTrainer class whose fit walks a DataLoader."""
    raise NotImplementedError

def _test_cx28():
    LoaderTrainer = cx28_make_trainer_with_fit()

    # Tiny regression task with TensorDataset + DataLoader.
    t.manual_seed(0)
    N = 32
    x_data = t.randn(N, 1)
    y_data = 2.5 * x_data - 0.5 + 0.01 * t.randn(N, 1)
    ds = TensorDataset(x_data, y_data)
    loader = DataLoader(ds, batch_size=8, shuffle=False)
    assert len(loader) == 4, f'sanity: 32/8 = 4 batches; got {len(loader)}'

    model = nn.Linear(1, 1)
    opt = t.optim.SGD(model.parameters(), lr=0.05)
    loss_fn = nn.MSELoss()
    trainer = LoaderTrainer(model, opt, loss_fn, loader)

    # Case A: initial state.
    assert trainer.step == 0
    assert trainer.epoch == 0
    assert trainer.history == []
    assert trainer.train_loader is loader

    # Case B: fit walks the right number of batches.
    n_epochs = 5
    history = trainer.fit(n_epochs)
    expected_steps = n_epochs * len(loader)  # 5 * 4 = 20.
    assert trainer.step == expected_steps, (
        f'step should be {expected_steps} after {n_epochs} epochs of {len(loader)} batches; '
        f'got {trainer.step}'
    )
    assert trainer.epoch == n_epochs, f'epoch should be {n_epochs}; got {trainer.epoch}'
    assert len(trainer.history) == expected_steps
    assert history is trainer.history or history == trainer.history

    # Case C: history contains floats (per-batch losses).
    assert all(isinstance(v, float) for v in trainer.history), 'history must store floats'

    # Case D: loss decreases across epochs (regression converges).
    first_epoch_avg = sum(trainer.history[:len(loader)]) / len(loader)
    last_epoch_avg = sum(trainer.history[-len(loader):]) / len(loader)
    assert last_epoch_avg < first_epoch_avg, (
        f'avg loss should decrease across epochs: first={first_epoch_avg:.4f} '
        f'last={last_epoch_avg:.4f}'
    )

    # Case E: batch-size invariant — fit a SECOND trainer with N=30, B=8 (uneven).
    # 30 / 8 = 3 full batches of 8 + 1 partial batch of 6 = 4 batches; step should be n_epochs*4.
    x2 = t.randn(30, 1)
    y2 = 2.5 * x2 - 0.5
    ds2 = TensorDataset(x2, y2)
    loader2 = DataLoader(ds2, batch_size=8, shuffle=False, drop_last=False)
    assert len(loader2) == 4, f'sanity: ceil(30/8)=4; got {len(loader2)}'
    model2 = nn.Linear(1, 1)
    opt2 = t.optim.SGD(model2.parameters(), lr=0.01)
    tr2 = LoaderTrainer(model2, opt2, nn.MSELoss(), loader2)
    tr2.fit(3)
    assert tr2.step == 3 * 4, f'uneven batches: step should be 12; got {tr2.step}'
    # Verify last batch was the partial one — we can't easily inspect it post hoc, but
    # step count being right implies fit iterated all 4 batches per epoch.
    assert tr2.epoch == 3
    _dd_passed.add('cx28')

_test_cx28()

<details><summary>Show solution — cx28</summary>

```python
def cx28_make_trainer_with_fit():
    class LoaderTrainer:
        def __init__(self, model, optimizer, loss_fn, train_loader):
            self.model = model
            self.optimizer = optimizer
            self.loss_fn = loss_fn
            self.train_loader = train_loader
            self.step = 0
            self.epoch = 0
            self.history = []

        def training_step(self, batch):
            x, y = batch
            logits = self.model(x)
            loss = self.loss_fn(logits, y)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            self.step += 1
            self.history.append(loss.item())
            return loss

        def fit(self, n_epochs):
            # Atom B (dataloader-batching): one training_step per batch yielded.
            for _ in range(n_epochs):
                for batch in self.train_loader:
                    self.training_step(batch)
                self.epoch += 1
            return self.history

    return LoaderTrainer
```

`for batch in loader` is what makes DataLoader composable with the trainer skeleton — the loader handles shuffling, batching, and the partial-final-batch edge case; the trainer just handles 'apply one optimizer step per batch'. If you set `drop_last=True` the partial batch is skipped and `len(loader)` shrinks by 1.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx28'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx28',
        'subtopics': ["Trainer: Trainer class skeleton", "PyTorch: DataLoader batching"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()